# Phase 13 — Final Model Packaging, Inference Pipeline & Deployment

## Objective

Convert the final Phase 11/12 model into a clean, reusable ML inference system.

By the end of this phase we will have:

```text
Raw customer-level features
        ↓
Saved preprocessing pipeline
        ↓
Saved tuned ML model
        ↓
Prediction probability
        ↓
Business threshold
        ↓
Purchase / No Purchase
```

This phase focuses on **productionization**, not further model experimentation.

### Important
The model and threshold are already frozen from previous phases. We do not tune them here.

## 1. Production architecture

The final system has four layers:

### Layer 1 — Feature Input
Receives the same customer-level features used during training.

### Layer 2 — Preprocessing
Loads the exact preprocessing pipeline fitted during training.

### Layer 3 — Model
Loads the frozen tuned model.

### Layer 4 — Prediction
Returns:
- purchase probability
- predicted class
- model version
- threshold used

This prevents training-time preprocessing and inference-time preprocessing from becoming inconsistent.

In [ ]:
from pathlib import Path
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
DEPLOY_DIR = BASE_DIR / "deployment"

DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", BASE_DIR)

## 2. Load the frozen model configuration

The model was selected in Phase 11 and the operating threshold was selected using validation data in Phase 12.

In [ ]:
with open(RESULTS_DIR / "phase11_best_params.json", "r") as f:
    phase11_config = json.load(f)

with open(RESULTS_DIR / "phase12_final_evaluation_config.json", "r") as f:
    phase12_config = json.load(f)

MODEL_NAME = phase11_config["model"]
THRESHOLD = float(phase12_config["threshold"])

MODEL_PATH = MODELS_DIR / (
    MODEL_NAME.lower().replace(" ", "_") +
    "_phase11_tuned.joblib"
)

PREPROCESSOR_PATH = MODELS_DIR / "preprocessor_phase11_tuned.joblib"

print("Model:", MODEL_NAME)
print("Threshold:", THRESHOLD)
print("Model path:", MODEL_PATH)

## 3. Load model and preprocessing pipeline

In [ ]:
model = joblib.load(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)

print("Model loaded:", type(model).__name__)
print("Preprocessor loaded:", type(preprocessor).__name__)

## 4. Define the expected feature schema

A production model should explicitly define which columns it expects.

This prevents silent failures when:
- a column is missing,
- a column is renamed,
- an unexpected column is supplied,
- data types change.

In [ ]:
train_reference = pd.read_parquet(
    PROCESSED_DIR / "train_phase5.parquet"
)

EXPECTED_FEATURES = [
    c for c in train_reference.columns
    if c != "customer_id"
]

FEATURE_SCHEMA = {
    "feature_count": len(EXPECTED_FEATURES),
    "features": EXPECTED_FEATURES,
}

print("Expected feature count:", len(EXPECTED_FEATURES))
print(EXPECTED_FEATURES)

## 5. Build the inference function

The inference function performs exactly the same sequence every time:

```text
input validation
→ preprocessing
→ probability prediction
→ thresholding
→ structured output
```

In [ ]:
def validate_input(X):
    if not isinstance(X, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    missing = [
        c for c in EXPECTED_FEATURES
        if c not in X.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required features: {missing}"
        )

    extra = [
        c for c in X.columns
        if c not in EXPECTED_FEATURES and c != "customer_id"
    ]

    if extra:
        print("Warning: extra columns ignored:", extra)

    return True


def predict_customers(X):
    validate_input(X)

    X_model = X[
        [c for c in EXPECTED_FEATURES if c in X.columns]
    ].copy()

    start = time.perf_counter()

    X_transformed = preprocessor.transform(X_model)
    probabilities = model.predict_proba(X_transformed)[:, 1]

    predictions = (probabilities >= THRESHOLD).astype(int)

    latency = time.perf_counter() - start

    output = pd.DataFrame({
        "purchase_probability": probabilities,
        "prediction": predictions,
        "prediction_label": np.where(
            predictions == 1,
            "Purchase",
            "No Purchase"
        ),
    })

    if "customer_id" in X.columns:
        output.insert(
            0,
            "customer_id",
            X["customer_id"].values
        )

    output["model_name"] = MODEL_NAME
    output["threshold"] = THRESHOLD

    print(
        f"Prediction completed for {len(X):,} customers "
        f"in {latency:.4f} seconds."
    )

    return output

## 6. Test the inference pipeline on Phase 5 test data

This is an inference test only.

We are not changing the model or threshold.

In [ ]:
test_input = pd.read_parquet(
    PROCESSED_DIR / "test_phase5.parquet"
)

# Use a small sample for the demonstration.
sample_input = test_input.head(10).copy()

predictions = predict_customers(sample_input)

display(predictions)

## 7. Validate prediction output

Production inference should return:
- one prediction per input row,
- probabilities between 0 and 1,
- binary predictions,
- the frozen threshold.

In [ ]:
assert len(predictions) == len(sample_input)

assert predictions["purchase_probability"].between(
    0, 1
).all()

assert predictions["prediction"].isin([0, 1]).all()

assert (predictions["threshold"] == THRESHOLD).all()

print("Inference validation passed.")

## 8. Batch inference

For a large customer population, prediction should be performed in batches rather than creating one massive transformed matrix.

This reduces memory pressure.

In [ ]:
def batch_predict(X, batch_size=50_000):
    outputs = []

    for start in range(0, len(X), batch_size):
        batch = X.iloc[start:start + batch_size]
        outputs.append(predict_customers(batch))

    if outputs:
        return pd.concat(outputs, ignore_index=True)

    return pd.DataFrame()


batch_sample = test_input.head(100_000).copy()

batch_predictions = batch_predict(
    batch_sample,
    batch_size=10_000
)

print("Rows predicted:", len(batch_predictions))
display(batch_predictions.head())

## 9. Create a production prediction report

A useful downstream system needs more than a class label.

We produce:
- customer ID
- purchase probability
- predicted class
- threshold
- model version

In [ ]:
production_output = batch_predictions.copy()

production_output["model_version"] = "phase13-final-v1"

production_output = production_output[
    [
        c for c in [
            "customer_id",
            "purchase_probability",
            "prediction",
            "prediction_label",
            "threshold",
            "model_name",
            "model_version",
        ]
        if c in production_output.columns
    ]
]

display(production_output.head(10))

## 10. Identify high-confidence customers

This is a simple business interpretation layer.

It does not modify the model.

We divide customers into:
- High probability
- Medium probability
- Low probability

In [ ]:
def probability_band(p):
    if p >= 0.75:
        return "High"
    elif p >= 0.40:
        return "Medium"
    return "Low"

production_output["probability_band"] = (
    production_output["purchase_probability"]
    .apply(probability_band)
)

display(
    production_output["probability_band"]
    .value_counts()
    .rename_axis("band")
    .reset_index(name="customers")
)

## 11. Save batch predictions

This output can later be consumed by:
- a dashboard,
- a campaign system,
- an API,
- an analytics pipeline.

In [ ]:
prediction_file = DEPLOY_DIR / "customer_predictions_phase13.csv"

production_output.to_csv(
    prediction_file,
    index=False
)

print("Saved:", prediction_file)

## 12. Create a model manifest

A model registry/manifest should record exactly what was deployed.

In [ ]:
manifest = {
    "model_version": "phase13-final-v1",
    "project": "H&M 30-Day Customer Purchase Prediction",
    "model_name": MODEL_NAME,
    "model_artifact": str(MODEL_PATH),
    "preprocessor_artifact": str(PREPROCESSOR_PATH),
    "classification_threshold": THRESHOLD,
    "primary_metric": "PR-AUC",
    "prediction_horizon_days": 30,
    "test_used_for_threshold_selection": False,
    "test_used_for_model_selection": False,
    "images_used": False,
    "feature_count": len(EXPECTED_FEATURES),
}

with open(
    DEPLOY_DIR / "model_manifest.json",
    "w"
) as f:
    json.dump(manifest, f, indent=2)

display(pd.DataFrame([manifest]))

## 13. Create an inference configuration

This separates deployment configuration from the notebook itself.

In [ ]:
inference_config = {
    "model_version": "phase13-final-v1",
    "threshold": THRESHOLD,
    "batch_size": 50_000,
    "expected_feature_count": len(EXPECTED_FEATURES),
    "prediction_horizon_days": 30,
}

with open(
    DEPLOY_DIR / "inference_config.json",
    "w"
) as f:
    json.dump(inference_config, f, indent=2)

print("Inference configuration saved.")

## 14. Save the feature schema

In [ ]:
with open(
    DEPLOY_DIR / "feature_schema.json",
    "w"
) as f:
    json.dump(FEATURE_SCHEMA, f, indent=2)

print("Feature schema saved.")

## 15. Model artifact size

Tracking artifact size is useful when considering deployment environments.

In [ ]:
model_size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)
preprocessor_size_mb = PREPROCESSOR_PATH.stat().st_size / (1024 ** 2)

artifact_sizes = pd.DataFrame([
    {
        "artifact": "Model",
        "size_mb": model_size_mb
    },
    {
        "artifact": "Preprocessor",
        "size_mb": preprocessor_size_mb
    }
])

display(artifact_sizes)

## 16. Inference latency benchmark

We benchmark several batch sizes.

This is useful when deciding between:
- batch inference,
- REST API inference,
- asynchronous inference.

In [ ]:
benchmark_rows = []

benchmark_input = test_input.head(20_000).copy()

for batch_size in [1_000, 5_000, 10_000, 20_000]:
    start = time.perf_counter()

    result = batch_predict(
        benchmark_input,
        batch_size=batch_size
    )

    elapsed = time.perf_counter() - start

    benchmark_rows.append({
        "batch_size": batch_size,
        "rows": len(result),
        "total_seconds": elapsed,
        "rows_per_second": len(result) / elapsed,
    })

benchmark_df = pd.DataFrame(benchmark_rows)

display(benchmark_df)

## 17. Production data-quality checks

Before making predictions, production data should be checked for:
- missing required columns,
- duplicate customer IDs,
- impossible numeric values,
- unexpected categorical values,
- extreme missingness.

In [ ]:
def production_data_quality_report(X):
    report = {
        "rows": len(X),
        "columns": len(X.columns),
        "duplicate_customer_ids": (
            X["customer_id"].duplicated().sum()
            if "customer_id" in X.columns else None
        ),
        "missing_cells": int(X.isna().sum().sum()),
        "missing_percentage": float(
            X.isna().mean().mean() * 100
        ),
    }

    if "age" in X.columns:
        report["invalid_age_count"] = int(
            ((X.age < 0) | (X.age > 120)).sum()
        )

    return pd.DataFrame([report])


display(
    production_data_quality_report(sample_input)
)

## 18. Reproducibility information

A production ML system should preserve:
- random seed,
- model version,
- threshold,
- metric,
- feature count,
- prediction horizon.

In [ ]:
reproducibility = {
    "random_state": 42,
    "model_version": "phase13-final-v1",
    "model_name": MODEL_NAME,
    "threshold": THRESHOLD,
    "primary_metric": "PR-AUC",
    "horizon_days": 30,
    "feature_count": len(EXPECTED_FEATURES),
}

with open(
    DEPLOY_DIR / "reproducibility.json",
    "w"
) as f:
    json.dump(reproducibility, f, indent=2)

display(pd.DataFrame([reproducibility]))

## 19. Deployment-ready directory

After this phase, the deployment artifacts are organized as:

```text
deployment/
├── customer_predictions_phase13.csv
├── model_manifest.json
├── inference_config.json
├── feature_schema.json
└── reproducibility.json

models/
├── <final_model>_phase11_tuned.joblib
└── preprocessor_phase11_tuned.joblib
```

This separates model artifacts from experimentation notebooks.

In [ ]:
print("Deployment directory contents:")
for p in sorted(DEPLOY_DIR.iterdir()):
    print(" -", p.name)

## 20. Optional API structure

For deployment, the same `predict_customers()` logic can be placed behind an API.

Example architecture:

```text
Client
  ↓
FastAPI / REST API
  ↓
Input validation
  ↓
Feature schema validation
  ↓
Saved preprocessor
  ↓
Saved model
  ↓
Probability + threshold
  ↓
JSON response
```

A typical response would contain:

```json
{
  "customer_id": "...",
  "purchase_probability": 0.73,
  "prediction": 1,
  "prediction_label": "Purchase",
  "threshold": 0.42,
  "model_version": "phase13-final-v1"
}
```

The important design principle is that the API should **load the frozen artifacts**, not retrain the model.

# Phase 13 — Final Project Status

The complete ML lifecycle is now represented:

| Phase | Component |
|---|---|
| 1 | Problem definition + data audit |
| 2 | Data cleaning + quality |
| 3 | EDA |
| 4 | Feature engineering |
| 5 | Target + temporal split |
| 6 | ML preprocessing + PCA |
| 7 | Baseline + bias/variance |
| 8 | Multiple model selection |
| 9 | Gradient/optimization experiments |
| 10 | Ensemble learning |
| 11 | Hyperparameter tuning + temporal CV |
| 12 | Final evaluation + error analysis + explainability |
| **13** | **Model packaging + inference + deployment** |

## Placement-ready project narrative

> **Built an end-to-end customer purchase prediction system using the H&M transaction dataset, covering temporal feature engineering, leakage-safe preprocessing, dimensionality reduction, baseline modeling, ensemble learning, gradient optimization, time-aware hyperparameter tuning, threshold optimization, error analysis, explainability, and deployment-ready inference.**

## Next

**Phase 14 — Final Project Report + CV Bullets + Interview Grilling**

This final phase will turn the technical work into:
- 3 strong CV bullets,
- project objective,
- tech stack,
- quantified results,
- architecture explanation,
- likely interviewer questions,
- ML theory questions,
- project-specific grilling,
- and concise answers for placements.